06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [2]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch

# WORKING WITH 
datasetPath = 'data/clothDataset_5_.csv'

cloth_info = pd.read_csv(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (42, 326)
cloth_info: 
    frame         x0        y0        z0         vx0        vy0        vz0  \
0       0   0.140004 -48.66936  24.85473   -19.49682  2408.4960 -1243.0510   
1       1   0.140004 -48.67611  24.86175   -19.50033  2408.2050 -1243.1320   
2       2   0.140004 -48.66178  24.85788   -19.49839  2408.5640 -1242.7920   
3       3   0.140004 -48.66509  24.86249   -19.50070  2408.1070 -1243.3150   
4       4   0.140004 -48.66097  24.85813   -19.49852  2408.3150 -1243.4690   
5       5   0.140004 -48.67393  24.85498   -19.49694  2409.1260 -1242.7850   
6       6   0.140004 -48.66834  24.84614   -19.49252  2408.3940 -1243.0310   
7       7   0.140004 -48.65874  24.86459   -19.50175  2408.1170 -1243.1090   
8       8 -20.740620 -45.32977  21.91692   853.18230  2281.2300 -1060.2820   
9       9 -62.632640 -25.51987  24.39955  3023.39500  1332.6920 -1271.4930   
10     10 -64.099960 -21.81283  21.40502  3252.25300  1042.7880 -1116.8080   
11     11  -5.190634 -4

In [ ]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=25):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        if isinstance(csv_data, str) and "frame,x0" in csv_data:
            self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        else:
            self.data = pd.read_csv(csv_data)
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data) - 1  # El ultimo frame NO tiene siguiente frame

    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_tensor(idx + 1) # TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(datasetPath)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break

Batch Shape: torch.Size([4, 25, 13])
tensor([[[ 6.6455e+01, -1.8690e+01,  1.7889e+01,  ...,  3.4028e+38,
           7.5000e-01,  0.0000e+00],
         [ 5.0871e+01, -2.7232e+00,  4.5383e+01,  ...,  3.4028e+38,
           1.0000e+00,  2.5000e-01],
         [ 6.9248e+01, -1.9779e+01,  4.2709e+01,  ...,  3.4028e+38,
           1.0000e+00,  0.0000e+00],
         ...,
         [ 7.9570e+00,  2.7588e+01, -5.0145e+01,  ...,  3.4028e+38,
           0.0000e+00,  7.5000e-01],
         [ 1.4000e-01,  5.1330e+01, -2.5139e+01,  ...,  0.0000e+00,
           2.5000e-01,  1.0000e+00],
         [ 1.3999e-01,  5.1330e+01, -5.0139e+01,  ...,  0.0000e+00,
           0.0000e+00,  1.0000e+00]],

        [[ 1.4000e-01, -4.8674e+01,  2.4855e+01,  ...,  3.4028e+38,
           7.5000e-01,  0.0000e+00],
         [ 1.4001e-01, -2.3673e+01,  4.9862e+01,  ...,  3.4028e+38,
           1.0000e+00,  2.5000e-01],
         [ 1.4001e-01, -4.8678e+01,  4.9856e+01,  ...,  3.4028e+38,
           1.0000e+00,  0.0000e+00],
  

In [ ]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLu()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas
model = MyModule(num_inputs=dataloader.shape[1], num_hidden=128, num_outputs=1)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    for batch_t, batch_t1 in dataloader:
        optimizer.zero_grad()
        pred = model(batch_t)
        loss = criterion(pred, batch_t1)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


In [ ]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
